# Deep dive 10/13: `satisfaction_surveys`

Dictionary declares 250,000 rows, Source: Contact Center, Partition: daily. Every column, Bronze and Silver.

`table_specs.py`'s comment makes a specific claim worth verifying directly rather than trusting: **`main_score` is described as "a 1-7 integer scale spanning CSAT/NPS/CES"**, but the dictionary itself describes it differently per survey type -- 1-5 for CSAT, 0-10 for NPS (CES isn't given an explicit range in the dictionary text). Section 7 checks the real observed range of `main_score`, broken out by `survey_type`, to see which description -- if either -- actually matches the data.

Dictionary NOT NULL: `survey_id`, `survey_date`, `process_date`, `customer_id`, `survey_type`, `send_channel`, `main_score` -- **7 columns**. `contracts.py`'s current `required` tuple has 6 -- **`send_channel` is missing**, checked in section 3 as a fix candidate. Nullable: `interaction_id`, `agent_id` (both nullable FKs -- not every survey ties back to a specific interaction or agent), `nps_category`, `question_1_text`/`question_1_response`, `question_2_text`/`question_2_response`, `question_3_text`/`question_3_response`, `open_comments`, `comment_sentiment`, `response_time_hours`, `campaign_response_rate`.

Since `call_center_interactions` is already checked, section 11 reuses the cross-table consistency pattern from `call_transcripts`: when `interaction_id` is populated, does the survey's `customer_id`/`agent_id` actually match the parent interaction's own values?

Run top to bottom. Checklist near the end; free cells after that.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver side by side (excluding long text columns)

In [ ]:
sample_ids = q("SELECT survey_id FROM bronze.satisfaction_surveys ORDER BY survey_id LIMIT 20")["survey_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

cols = ("survey_id, survey_date, process_date, interaction_id, customer_id, agent_id, survey_type, "
        "send_channel, main_score, nps_category, question_1_response, question_2_response, "
        "question_3_response, comment_sentiment, response_time_hours, campaign_response_rate")
print("Bronze:")
display(q(f"SELECT {cols} FROM bronze.satisfaction_surveys WHERE survey_id IN ({ids_sql}) ORDER BY survey_id"))
print("Silver:")
display(q(f"SELECT {cols} FROM silver.fact_satisfaction_surveys WHERE survey_id IN ({ids_sql}) ORDER BY survey_id"))

## 1. Column inventory -- names AND types, Bronze vs. Silver

In [ ]:
expected_silver_columns = [
    "survey_id", "survey_date", "process_date", "interaction_id", "customer_id", "agent_id",
    "survey_type", "send_channel", "main_score", "nps_category", "question_1_text",
    "question_1_response", "question_2_text", "question_2_response", "question_3_text",
    "question_3_response", "open_comments", "comment_sentiment", "response_time_hours",
    "campaign_response_rate",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_satisfaction_surveys'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_satisfaction_surveys:", missing or "none")
print("Extra in silver.fact_satisfaction_surveys:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

In [ ]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='satisfaction_surveys' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_satisfaction_surveys' ORDER BY ordinal_position
"""))

## 2. Row-count reconciliation + dedup

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.satisfaction_surveys")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT survey_id) AS n FROM bronze.satisfaction_surveys")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_satisfaction_surveys")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT survey_id) AS n FROM silver.fact_satisfaction_surveys")["n"][0]

print(f"bronze.satisfaction_surveys:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct survey_id")
print(f"silver.fact_satisfaction_surveys: {silver_total:,} rows, {silver_distinct_pk:,} distinct survey_id")
print()
print("Dictionary declares 250,000 rows -- matches" if bronze_total == 250_000 else f"Dictionary declares 250,000 rows -- MISMATCH, Bronze has {bronze_total:,}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 7 dictionary-declared columns

Including `send_channel`, missing from `contracts.py`'s `required` tuple -- checked here as a fix candidate.

In [ ]:
declared_not_null = ["survey_id", "survey_date", "process_date", "customer_id", "survey_type", "send_channel", "main_score"]
currently_required = {"survey_id","survey_date","process_date","customer_id","survey_type","main_score"}

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.satisfaction_surveys
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_satisfaction_surveys WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "in_contracts_required": col in currently_required,
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

## 4. Nullable columns -- population rate, Bronze vs. Silver

In [ ]:
nullable_cols = ["interaction_id", "agent_id", "nps_category", "question_1_text", "question_1_response",
                  "question_2_text", "question_2_response", "question_3_text", "question_3_response",
                  "open_comments", "comment_sentiment", "response_time_hours", "campaign_response_rate"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.satisfaction_surveys")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.satisfaction_surveys
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_satisfaction_surveys WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

## 5. Domain checks -- `survey_type`, `send_channel`, `nps_category`

In [ ]:
# survey_type -- dictionary: CSAT, NPS, CES
print("Bronze:")
display(q("SELECT survey_type, count(*) AS n FROM bronze.satisfaction_surveys GROUP BY 1 ORDER BY n DESC"))
# send_channel -- dictionary: Email, SMS, IVR, App, Web
print("send_channel:")
display(q("SELECT send_channel, count(*) AS n FROM bronze.satisfaction_surveys GROUP BY 1 ORDER BY n DESC"))
# nps_category -- dictionary: Promoter, Passive, Detractor
print("nps_category:")
display(q("SELECT nps_category, count(*) AS n FROM bronze.satisfaction_surveys GROUP BY 1 ORDER BY n DESC"))

## 6. FK integrity -- `interaction_id`/`agent_id` (nullable), `customer_id` (required), reproduced directly

In [ ]:
fk_checks = [
    ("interaction_id", "silver.fact_call_center_interactions", "interaction_id"),
    ("customer_id", "silver.dim_customers", "customer_id"),
    ("agent_id", "silver.dim_service_agents", "agent_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_satisfaction_surveys WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_satisfaction_surveys t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

## 7. `main_score` -- real observed range BY `survey_type`

`table_specs.py`'s comment claims "a 1-7 integer scale spanning CSAT/NPS/CES." The dictionary instead says 1-5 for CSAT and 0-10 for NPS (no explicit range given for CES). Checking which description -- if either -- the real data actually matches.

In [ ]:
q("""
    SELECT survey_type, count(*) AS n, min(main_score) AS min_score, max(main_score) AS max_score,
           count(DISTINCT main_score) AS distinct_values
    FROM silver.fact_satisfaction_surveys
    GROUP BY 1
    ORDER BY 1
""")

In [ ]:
# Full distribution per survey_type, to see the actual shape rather than just min/max.
for t in q("SELECT DISTINCT survey_type AS v FROM silver.fact_satisfaction_surveys")["v"]:
    print(f"{t}:")
    display(q(f"SELECT main_score, count(*) AS n FROM silver.fact_satisfaction_surveys WHERE survey_type = '{t}' GROUP BY 1 ORDER BY 1"))

## 8. `nps_category` -- does it line up with `main_score` the standard NPS way (Promoter/Passive/Detractor)?

In [ ]:
q("""
    SELECT nps_category, min(main_score) AS min_score, max(main_score) AS max_score, count(*) AS n
    FROM silver.fact_satisfaction_surveys
    WHERE survey_type = 'NPS' AND nps_category IS NOT NULL
    GROUP BY 1
    ORDER BY min_score
""")

## 9. `question_1_response`/`question_2_response`/`question_3_response` -- range sanity (dictionary: 1-5 each)

In [ ]:
q("""
    SELECT
        min(question_1_response) AS q1_min, max(question_1_response) AS q1_max,
        min(question_2_response) AS q2_min, max(question_2_response) AS q2_max,
        min(question_3_response) AS q3_min, max(question_3_response) AS q3_max,
        count(*) FILTER (WHERE question_1_response NOT BETWEEN 1 AND 5) AS q1_out_of_range,
        count(*) FILTER (WHERE question_2_response NOT BETWEEN 1 AND 5) AS q2_out_of_range,
        count(*) FILTER (WHERE question_3_response NOT BETWEEN 1 AND 5) AS q3_out_of_range
    FROM silver.fact_satisfaction_surveys
""")

## 10. `response_time_hours`/`campaign_response_rate` -- range sanity

In [ ]:
q("""
    SELECT
        typeof(response_time_hours) AS rt_type, typeof(campaign_response_rate) AS crr_type,
        min(response_time_hours) AS min_rt, max(response_time_hours) AS max_rt,
        min(campaign_response_rate) AS min_crr, max(campaign_response_rate) AS max_crr,
        count(*) FILTER (WHERE response_time_hours < 0) AS negative_rt,
        count(*) FILTER (WHERE campaign_response_rate < 0 OR campaign_response_rate > 100) AS crr_out_of_range
    FROM silver.fact_satisfaction_surveys
    GROUP BY ALL
""")

## 11. Cross-table consistency -- when `interaction_id` is populated, does `customer_id`/`agent_id` match the parent interaction's own?

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE s.customer_id = i.customer_id) AS customer_id_matches,
        count(*) FILTER (WHERE s.customer_id <> i.customer_id) AS customer_id_mismatches,
        count(*) FILTER (WHERE s.agent_id IS NOT NULL AND i.agent_id IS NOT NULL AND s.agent_id = i.agent_id) AS agent_id_matches,
        count(*) FILTER (WHERE s.agent_id IS NOT NULL AND i.agent_id IS NOT NULL AND s.agent_id <> i.agent_id) AS agent_id_mismatches
    FROM silver.fact_satisfaction_surveys s
    JOIN silver.fact_call_center_interactions i ON i.interaction_id = s.interaction_id
    WHERE s.interaction_id IS NOT NULL
""")

## 12. `comment_sentiment` -- domain check

In [ ]:
q("SELECT comment_sentiment, count(*) AS n FROM bronze.satisfaction_surveys GROUP BY 1 ORDER BY n DESC")

## 13. `survey_date`/`process_date` -- cast correctness, window check

In [ ]:
q("""
    SELECT
        typeof(survey_date) AS survey_date_type, typeof(process_date) AS process_date_type,
        min(survey_date) AS earliest_survey, max(survey_date) AS latest_survey,
        min(process_date) AS earliest_process, max(process_date) AS latest_process,
        count(*) FILTER (WHERE process_date <> CAST(survey_date AS DATE)) AS process_date_differs
    FROM silver.fact_satisfaction_surveys
    GROUP BY ALL
""")

In [ ]:
q("""
    SELECT
        count(*) FILTER (WHERE survey_date < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE survey_date > TIMESTAMP '2026-06-17 23:59:59') AS after_window
    FROM silver.fact_satisfaction_surveys
""")

## Checklist

- [ ] Column inventory (1): all 19 Silver columns present, types correct
- [ ] Row reconciliation (2): 250,000 rows as the dictionary declares, dedup clean
- [ ] NOT NULL audit (3): all 7 declared columns populated -- note `send_channel` fix candidate
- [ ] Nullable columns (4): population rates make sense
- [ ] Domain checks (5): `survey_type`/`send_channel`/`nps_category` real values
- [ ] FK integrity (6): all three FKs reproduced directly
- [ ] `main_score` (7): resolves whether the real per-survey_type range matches the dictionary, the code comment, both, or neither
- [ ] `nps_category` (8): lines up with `main_score` the standard NPS way
- [ ] Question responses (9): within 1-5
- [ ] `response_time_hours`/`campaign_response_rate` (10): no negatives, rate within 0-100
- [ ] Cross-table consistency (11): survey `customer_id`/`agent_id` matches the parent interaction's, when populated
- [ ] `comment_sentiment` (12): domain sanity
- [ ] Dates (13): window check

In [ ]:
con.close()
print("closed")